In [ ]:
# Install/load
install.packages(c("MatchIt", "cobalt", "readr")) # run once
library(MatchIt)
library(cobalt)
library(readr)


In [ ]:
lupus_data <- read_csv("lupus_matchit.csv")


In [ ]:
perform_matching <- function(input_data, 
                             match_ratio        = 10, 
                             exact_vars         = c("sex", "ancestry"),
                             plot_output_folder = NULL,   # where to save plots
                             plot_label         = NULL,   # used in title/filename
                             save_plots         = TRUE) {
  
  if (is.null(plot_label)) {
    plot_label <- "cohort"
  }
  
  m.out <- MatchIt::matchit(
    case ~ age + sex + ancestry,
    data     = input_data,
    method   = "nearest",
    exact    = exact_vars,
    ratio    = match_ratio,
    distance = "mahalanobis"
  )
  
  message("--- Matching Summary ---")
  print(summary(m.out))
  
  message("\n--- Balance Plot (Love Plot) ---")
  
  tryCatch({
    lp <- cobalt::love.plot(m.out)
    
    # Add title/subtitle if ggplot
    if (inherits(lp, "ggplot")) {
      lp <- lp +
        ggplot2::labs(
          title    = paste0("Love plot: ", plot_label),
          subtitle = paste0(match_ratio, ":1 matching (controls:cases)")
        )
    }
    
    # Show it in the R session
    print(lp)
    
    # Save to file if requested
    if (!is.null(plot_output_folder) && isTRUE(save_plots)) {
      dir.create(plot_output_folder, showWarnings = FALSE, recursive = TRUE)
      
      plot_file <- file.path(
        plot_output_folder,
        paste0("love_plot_", plot_label, ".png")
      )
      message("Saving love plot to: ", plot_file)
      
      if (inherits(lp, "ggplot")) {
        ggplot2::ggsave(
          filename = plot_file,
          plot     = lp,
          width    = 8,
          height   = 6,
          dpi      = 300
        )
      } else {
        png(plot_file, width = 1200, height = 800, res = 150)
        cobalt::love.plot(m.out)
        dev.off()
      }
    }
  }, error = function(e) {
    message("Could not generate/save love.plot. Error: ", e$message)
  })
  
  matched_data <- MatchIt::match.data(m.out)
  return(matched_data)
}


In [ ]:
lupus <- perform_matching(
  input_data        = lupus_data,          # your df with case/age/sex/ancestry
  match_ratio       = 10,                   # e.g. 5 controls per case
  exact_vars        = c("sex", "ancestry"),
  plot_output_folder = "matchit_love_plots",   # folder where PNG goes
  plot_label        = "lupus",              # used in plot title + filename
  save_plots        = TRUE
)

readr::write_csv(lupus, "lupus_matched_10.csv")

In [ ]:
n_cases    <- sum(lupus_data$case == 1)
n_controls <- sum(lupus_data$case == 0)

max_ratio <- floor(n_controls / n_cases)

lupus2 <- perform_matching(
  input_data        = lupus_data,          # your df with case/age/sex/ancestry
  match_ratio       = max_ratio,                   # e.g. 5 controls per case
  exact_vars        = c("sex", "ancestry"),
  plot_output_folder = "matchit_love_plots",   # folder where PNG goes
  plot_label        = "lupus_ALL",              # used in plot title + filename
  save_plots        = TRUE
)

readr::write_csv(lupus2, "lupus_matched_ALL.csv")

In [ ]:
max_ratio